# Stage 7 - cell tracking

This notebook is a thin runner for the public Stage 7 API. Tracking and graph optimization logic remain in `src/`.

Graph modes:

- `disabled`: run and save only the provisional Stage 7 tracker.
- `shadow`: run the selected graph backend and save its diagnostics, but keep provisional tracks as `tracks.csv`.
- `apply`: save graph-optimized tracks to the normal Stage 7 output directory. These files affect Stages 8-12.

After an apply-mode Stage 7 run, rerun Stage 8 before Stage 9 because Stage 9 loads the latest saved Stage 8 tracks.

## 1. Sample and path configuration

In [ ]:
from src.io import PipelinePaths

SAMPLE_ID = "44b6_0113de3b"
paths = PipelinePaths.discover()
sample_path = paths.sample_zarr(SAMPLE_ID)

print("Sample:", SAMPLE_ID)
print("Stage 7 output directory:", paths.stage7_tracking)

## 2. Graph-tracking configuration

Allowed `GRAPH_MODE` values are `disabled`, `shadow`, and `apply`. Allowed `GRAPH_ALGORITHM` values are `pairwise` and `windowed_4d`. The settings below intentionally select the current apply-mode 4D research experiment; they do not change the library default.

In [ ]:
from src.api import run_cell_tracking, GraphTrackingConfig, FourDGraphConfig

GRAPH_MODE = "apply"              # disabled, shadow, apply
GRAPH_ALGORITHM = "windowed_4d"  # pairwise, windowed_4d

# Compact experiment controls for the windowed 4D backend.
WINDOW_SIZE = 7
MAXIMUM_GAP_FRAMES = 2
SOLVER_TIME_LIMIT_SECONDS = 30.0
ITERATIVE_FALLBACK_ITERATIONS = 5
SHOW_DETAILED_DIAGNOSTICS = True
SAVE_DEBUG_NPZ = False

if WINDOW_SIZE < 3 or WINDOW_SIZE % 2 == 0:
    raise ValueError("WINDOW_SIZE must be an odd integer of at least three")

four_d_config = FourDGraphConfig(
    window_size=WINDOW_SIZE,
    lookback_frames=WINDOW_SIZE // 2,
    lookahead_frames=WINDOW_SIZE // 2,
    maximum_gap_frames=MAXIMUM_GAP_FRAMES,
    solver_time_limit_seconds=SOLVER_TIME_LIMIT_SECONDS,
    iterative_solver_iterations=ITERATIVE_FALLBACK_ITERATIONS,
    save_detailed_debug_artifacts=SAVE_DEBUG_NPZ,
)
graph_config = GraphTrackingConfig(
    mode=GRAPH_MODE,
    algorithm=GRAPH_ALGORITHM,
    four_d=four_d_config,
)
graph_config

## 3. Load processed inputs

In [ ]:
from src.io import load_processed_dataset_inputs

inputs = load_processed_dataset_inputs(SAMPLE_ID, paths=paths)
time_frames = list(inputs.time_frames)
print(f"Loaded {len(time_frames)} frames and {sum(map(len, time_frames)):,} detections")

## 4. Run Stage 7

The default 20-frame `windowed_4d` experiment can be slow. Execute this cell deliberately; notebook validation does not run it.

In [ ]:
tracking = run_cell_tracking(
    list(inputs.time_frames),
    sample_id=SAMPLE_ID,
    graph_config=graph_config,
)

## 5. Save Stage 7 outputs

In apply mode this overwrites the normal Stage 7 files, including `tracks.csv`, with the optimized result consumed by Stage 8.

In [ ]:
from src.io import save_tracking_result

save_tracking_result(tracking, paths.stage7_tracking)
graph_metadata = tracking.metadata.get("graph_tracking", {})
optimized_output_applied = (
    graph_metadata.get("mode") == "apply"
    and bool(graph_metadata.get("enabled", False))
)
assignment_changes = graph_metadata.get(
    "assignment_changes",
    tracking.summary.get("graph4d_assignment_changes", tracking.summary.get("graph_assignment_changes", 0)),
)
selected_gaps = graph_metadata.get(
    "selected_gap_edges", tracking.summary.get("graph4d_selected_gap_edges", 0)
)

print("Saved Stage 7 output directory:", paths.stage7_tracking)
print("Graph mode:", graph_metadata.get("mode", GRAPH_MODE))
print("Graph algorithm:", graph_metadata.get("algorithm", GRAPH_ALGORITHM))
print("Optimized output applied:", optimized_output_applied)
print("Final track count:", tracking.tracks["track_id"].nunique())
print("Graph assignment changes:", assignment_changes)
print("Selected gap edges:", selected_gaps)
print("Boundary entries:", graph_metadata.get("optimized_boundary_entries", graph_metadata.get("graph_supported_entry_count", 0)))
print("Boundary exits:", graph_metadata.get("optimized_boundary_exits", graph_metadata.get("graph_supported_exit_count", 0)))
print("Validation status:", graph_metadata.get("validation_status", "not applicable"))
print("Runtime by phase:")
for phase, seconds in graph_metadata.get("runtime_seconds_by_phase", {}).items():
    print(f"  {phase}: {seconds:.3f} s")
print("Stage 8 will consume the tracks now saved at:", paths.stage7_tracking / "tracks.csv")

## 6. Inspect summary and metadata

In [ ]:
import pandas as pd
from IPython.display import display

graph_metadata = tracking.metadata.get("graph_tracking", {})
metadata_fields = [
    "mode", "algorithm", "provisional_track_count", "optimized_track_count",
    "observation_node_count", "spatial_edge_count", "temporal_candidate_count",
    "component_count", "exact_milp_component_count",
    "iterative_fallback_component_count", "assignment_changes",
    "selected_adjacent_edges", "selected_gap_edges",
    "optimized_boundary_entries", "optimized_boundary_exits",
    "solver_failures", "validation_status",
]
metadata_preview = {key: graph_metadata.get(key) for key in metadata_fields}
metadata_preview["runtime_seconds_by_phase"] = graph_metadata.get("runtime_seconds_by_phase", {})

print("Tracking summary")
display(pd.Series(tracking.summary, dtype=object, name="value"))
print("Graph-tracking metadata")
display(pd.Series(metadata_preview, dtype=object, name="value"))

## 7. Inspect 4D diagnostics

All seven tables have stable empty schemas, so these cells also work in disabled and pairwise modes.

In [ ]:
diagnostic_tables = [
    ("graph4d_window_summary", tracking.graph4d_window_summary, ["component_id", "window_id"]),
    ("graph4d_component_summary", tracking.graph4d_component_summary, ["component_id"]),
    ("graph4d_temporal_edges", tracking.graph4d_temporal_edges, ["source_frame", "source_detection_index", "target_frame", "target_detection_index"]),
    ("graph4d_assignment_changes", tracking.graph4d_assignment_changes, ["source_frame", "source_detection_index", "target_frame", "target_detection_index"]),
    ("graph4d_boundary_events", tracking.graph4d_boundary_events, ["frame", "detection_index", "event_type"]),
    ("graph4d_solver_diagnostics", tracking.graph4d_solver_diagnostics, ["component_id", "window_id"]),
    ("graph4d_track_id_map", tracking.graph4d_track_id_map, ["provisional_track_id", "optimized_track_id"]),
]

for table_name, table, preferred_sort in diagnostic_tables:
    print(f"{table_name}: {len(table):,} rows")
    if SHOW_DETAILED_DIAGNOSTICS:
        sort_columns = [column for column in preferred_sort if column in table.columns]
        preview = table.sort_values(sort_columns, kind="mergesort") if sort_columns else table
        display(preview.head(25))

In [ ]:
temporal_edges = tracking.graph4d_temporal_edges
preferred_edge_columns = [
    "source_frame", "target_frame", "source_detection_index",
    "target_detection_index", "provisional_selected", "optimized_selected",
    "frame_gap", "unary_cost", "motion_cost", "graph_cost",
    "persistent_relation_cost", "total_effective_cost", "component_id",
    "window_id", "solver_type", "changed_from_provisional",
]
edge_columns = [column for column in preferred_edge_columns if column in temporal_edges.columns]

if temporal_edges.empty:
    selected_changed = temporal_edges.copy()
    selected_gaps = temporal_edges.copy()
else:
    selected_changed = temporal_edges.loc[
        temporal_edges["optimized_selected"].astype(bool)
        & temporal_edges["changed_from_provisional"].astype(bool)
    ]
    selected_gaps = temporal_edges.loc[
        temporal_edges["optimized_selected"].astype(bool)
        & (temporal_edges["frame_gap"] > 1)
    ]

print(f"Selected changed edges: {len(selected_changed):,}")
display(selected_changed[edge_columns].head(50))
print(f"Selected gap edges: {len(selected_gaps):,}")
display(selected_gaps[edge_columns].head(50))

components = tracking.graph4d_component_summary
fallback_components = (
    components.loc[components["fallback_used"].astype(bool)]
    if not components.empty and "fallback_used" in components else components.copy()
)
solver_diagnostics = tracking.graph4d_solver_diagnostics
solver_failures = (
    solver_diagnostics.loc[~solver_diagnostics["success"].astype(bool)]
    if not solver_diagnostics.empty and "success" in solver_diagnostics else solver_diagnostics.copy()
)
print(f"Fallback components: {len(fallback_components):,}")
display(fallback_components.head(50))
print(f"Solver failures: {len(solver_failures):,}")
display(solver_failures.head(50))

## 8. Optional comparison of provisional and optimized changes

This summary uses only the saved 4D diagnostic tables; it does not rebuild or rerun the graph.

In [ ]:
edges = tracking.graph4d_temporal_edges
if edges.empty:
    comparison = {"graph_diagnostics_available": False}
else:
    rejected = edges.loc[
        edges["provisional_selected"].astype(bool)
        & ~edges["optimized_selected"].astype(bool)
    ]
    added = edges.loc[
        ~edges["provisional_selected"].astype(bool)
        & edges["optimized_selected"].astype(bool)
    ]
    rejected_sources = set(rejected["source_node"])
    added_sources = set(added["source_node"])
    rejected_targets = set(rejected["target_node"])
    added_targets = set(added["target_node"])
    boundary_events = tracking.graph4d_boundary_events
    comparison = {
        "graph_diagnostics_available": True,
        "provisional_edges_replaced": len(rejected),
        "match_to_different_match": len(rejected_sources & added_sources),
        "match_to_end": len(rejected_sources - added_sources),
        "end_to_match": len(added_sources - rejected_sources),
        "birth_to_matched": len(added_targets - rejected_targets),
        "selected_gap_edges": int(
            (edges["optimized_selected"].astype(bool) & (edges["frame_gap"] > 1)).sum()
        ),
        "graph_expanded_selected_candidates": int(
            (edges["optimized_selected"].astype(bool) & edges["graph_expanded"].astype(bool)).sum()
        ),
        "boundary_entries": int((boundary_events["event_type"] == "boundary_entry").sum()) if not boundary_events.empty else 0,
        "boundary_exits": int((boundary_events["event_type"] == "boundary_exit").sum()) if not boundary_events.empty else 0,
    }

display(pd.Series(comparison, dtype=object, name="count"))